# 19. Operating-Point-Matched Shared vs Residual Dual-Head Selector — Development Only

This notebook performs the calibration-fair comparison that the first dual-head experiment did not provide.

## Question

Was the residual dual-head representation intrinsically worse, or did the evaluated system merely operate at a more conservative point because:

- it inherited the shared selector's **τ = 0.70** threshold;
- epoch 1 was selected by intrinsic F1;
- epoch 2 had higher start recall but was never decoded;
- the entity MLP was retrained jointly with the constraint head?

## Protocol

1. Use only the original v5 development partitions.
2. Reproduce the residual dual-head training for exactly two epochs and save both checkpoints.
3. Sweep the same threshold grid for:
   - shared selector;
   - dual epoch 1;
   - dual epoch 2.
4. Match the dual candidate to the shared **τ = 0.70** trigger budget on the 200-example tuning split.
5. Select the dual checkpoint/threshold using tuning data only.
6. Evaluate the frozen matched pair once on the independent 200-example confirmation split.
7. Do not read, decode, or tune on the repaired test.

The main analysis is without Hard-v2 to isolate selector behavior. An optional secondary confirmation uses the same selected operating points with Hard-v2 and performs no additional tuning.


## Why F1 may be misaligned

The generation pipeline has confidence, compatibility, coverage, and realization gates. Some false activations can therefore be rejected or safely realized. A missed entity-start activation is often permanent. Consequently, a precision-heavy selector can have higher intrinsic F1 but worse downstream entity coverage.

This notebook reports both intrinsic calibration curves and generation metrics as functions of trigger volume.


In [ ]:
# 1. Bootstrap (Xet disabled BEFORE any HF import — lesson from 12b).
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!pip -q install "transformers>=4.44,<5" "huggingface_hub>=0.25,<1" nltk rouge-score accelerate sentencepiece sacremoses sacrebleu

import sys, json, pickle, random, time, shutil, subprocess, re, unicodedata, hashlib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive
drive.mount('/content/drive')
print('Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU runtime required'


In [ ]:
# 1.1 PyTorch Geometric.
try:
    import torch_geometric
    print('ok:', torch_geometric.__version__)
except Exception:
    tv = torch.__version__.split('+')[0]; cv = torch.version.cuda
    url = f'https://data.pyg.org/whl/torch-{tv}+cu{cv.replace(".", "")}.html'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric', 'torch-scatter', 'torch-sparse', '-f', url])
    import torch_geometric


In [ ]:
import glob
# 3. Development-only paths and configuration.
PROJECT_DIR = '/content/drive/MyDrive/kg_llm_project'
PROCESSED_DIR = f'{PROJECT_DIR}/baseline-bart-webnlg/processed'
CKPT_FUSION = f'{PROJECT_DIR}/fusion_only_outputs/checkpoints/fusion_only/model_best.pt'
CURRENT_SELECTOR_DIR = f'{PROJECT_DIR}/selector_eval_v5_earlystop'
CURRENT_SELECTOR_CKPT = f'{CURRENT_SELECTOR_DIR}/selector_best_v5.pt'
CURRENT_SPLIT_MANIFEST = f'{CURRENT_SELECTOR_DIR}/selector_dev_splits_v5.json'
OUTPUT_DIR = f'{PROJECT_DIR}/dual_head_operating_point_dev_v1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda'
MAX_GEN_LEN, MAX_INPUT_LEN, MAX_TARGET_LEN = 128, 256, 128
LOCK_MIN_DEPTH, ESCAPE_MARGIN = 2, 10.0

MODEL_DIM = 768
CONSTRAINT_BOTTLENECK = 256
CONSTRAINT_NUM_BASES = 8
CONSTRAINT_DROPOUT = 0.10

# Reproduce the original dual training through epoch 2, but save every epoch.
DUAL_EPOCHS = 2
TRAIN_BATCH_SIZE = 24
VAL_BATCH_SIZE = 16
HEAD_LR = 3e-4
SELECTOR_ENTITY_LR = 1e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NONE_WEIGHT = 0.30
DELTA_L2_WEIGHT = 1e-5
TRAIN_NONE_MLP = False
FORCE_RETRAIN = False

# Calibration sweep. Same grid for all selector variants.
TAU_GRID = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
SHARED_REFERENCE_TAU = 0.70
COMPATIBILITY_MARGIN = 8.0
MATCH_TOLERANCE_REL = 0.03
BOOTSTRAP_SAMPLES = 5000
RUN_TUNE_SWEEP = True
RUN_CONFIRMATION = True
RUN_HARD_V2_SECONDARY = True

for path in (f'{PROJECT_DIR}/fixed_ablation_common.py', PROCESSED_DIR, CKPT_FUSION, CURRENT_SELECTOR_CKPT):
    assert os.path.exists(path), f'Missing: {path}'

shutil.copy(f'{PROJECT_DIR}/fixed_ablation_common.py', '/content/fixed_ablation_common.py')
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
import fixed_ablation_common as fac
from fixed_ablation_common import (
    load_artifacts, FusionOnlyGNNModel, load_variant_checkpoint_resume,
    add_final_logits_bias, pad_kg_nodes, find_entity_token_spans,
)

from urllib.parse import quote
BART_LOCAL = '/content/bart-base-local'
os.makedirs(BART_LOCAL, exist_ok=True)
FILES = {
    'config.json': 1000,
    'vocab.json': 800000,
    'merges.txt': 400000,
    'tokenizer.json': 1000000,
    'model.safetensors': 500000000,
}
for filename, minimum_size in FILES.items():
    destination = os.path.join(BART_LOCAL, filename)
    if os.path.isfile(destination) and os.path.getsize(destination) >= minimum_size:
        continue
    url = f'https://huggingface.co/facebook/bart-base/resolve/main/{quote(filename)}?download=true'
    return_code = subprocess.run([
        'curl','-L','--fail','--retry','12','--retry-delay','5','--retry-all-errors',
        '--connect-timeout','30','--speed-time','90','--speed-limit','1024',
        '-C','-','-o',destination + '.part',url,
    ]).returncode
    if return_code != 0:
        if os.path.exists(destination + '.part'):
            os.remove(destination + '.part')
        return_code = subprocess.run(['curl','-L','--fail','-o',destination + '.part',url]).returncode
    assert return_code == 0 and os.path.getsize(destination + '.part') >= minimum_size
    os.replace(destination + '.part', destination)
    print('Downloaded', filename)

print('Output directory:', OUTPUT_DIR)
print('Protocol: development only; no repaired-test access.')


In [ ]:
# 3. Load data + frozen fusion model.
from transformers import BartTokenizer, BartForConditionalGeneration
tokenizer = BartTokenizer.from_pretrained(BART_LOCAL)
bart = BartForConditionalGeneration.from_pretrained(BART_LOCAL, local_files_only=True).to(DEVICE)
data, graphs, vocab = load_artifacts(PROCESSED_DIR)
num_relations = len(vocab['relation_vocab']) * 2
model = FusionOnlyGNNModel(bart, num_relations=num_relations).to(DEVICE)
model = load_variant_checkpoint_resume(model, CKPT_FUSION, DEVICE, strict=False)
model.eval()
for p in model.parameters(): p.requires_grad_(False)
print('frozen fusion model ready | splits:', {k: len(v) for k, v in data.items()})


In [ ]:
fusion_model = model
fusion_model.eval()
for p in fusion_model.parameters(): p.requires_grad_(False)
print('Frozen fusion backbone registered as fusion_model.')


In [ ]:
from difflib import SequenceMatcher
import sacrebleu

def _tp(t):
    if isinstance(t, dict): return str(t.get('subject','')), str(t.get('predicate','')), str(t.get('object',''))
    return str(t[0]), str(t[1]), str(t[2])

_TR = {'ø':'o','Ø':'o','æ':'ae','Æ':'ae','œ':'oe','Œ':'oe','ð':'d','Ð':'d','þ':'th','Þ':'th',
       'ł':'l','Ł':'l','ß':'ss','đ':'d','Đ':'d','ħ':'h','ı':'i','İ':'i','ŋ':'ng'}
_MONTHS = ['january','february','march','april','may','june','july','august','september','october','november','december']
_DET = re.compile(r'^(the|a|an)\s+')
_STOP = frozenset(('the a an and or but if then this that these those it its he she they them his her their '
                   'in on at of to by with for from as is are was were be been being there here also however').split())
def _sa(s):
    s = ''.join(_TR.get(c, c) for c in str(s))
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))
def _norm(s):
    s = _sa(str(s)).lower().strip().strip('"').strip("'")
    s = re.sub(r'[^a-z0-9 ]', ' ', s); s = re.sub(r'\s+', ' ', s).strip()
    return _DET.sub('', s)
def _wm(t, s):
    return bool(s) and re.search(r'(?<![a-z0-9])' + re.escape(s) + r'(?![a-z0-9])', t) is not None
def _dst(v):
    toks = set(); raw = _sa(str(v)).strip().strip('"').strip("'")
    m = re.match(r'^(\d{3,4})-(\d{1,2})-(\d{1,2})$', raw)
    if m:
        y, mo, d = map(int, m.groups()); toks.add(str(y))
        if 1 <= mo <= 12: toks.add(_MONTHS[mo-1]); toks.add(_MONTHS[mo-1][:3])
        toks.update({str(d), str(d).zfill(2)}); toks.update({f'{d}{sx}' for sx in ('st','nd','rd','th')})
    elif re.match(r'^\d{3,4}$', raw): toks.add(raw)
    return toks
def _dg(s): return re.sub(r'[^0-9]', '', str(s))
def grounding_score(prediction, triples):
    pn = _norm(prediction)
    forms, tokens, num = [], set(), set()
    for t in triples:
        s, p, o = _tp(t)
        for v in (s, o):
            n = _norm(v)
            if n: forms.append(n); tokens.update(n.split())
            tokens.update(_dst(v)); d = _dg(v)
            if d: num.add(d)
        tokens.update(_norm(re.sub(r'([a-z])([A-Z])', r'\1 \2', p)).split())
    fd = [f.replace(' ', '') for f in forms]
    found = sum(1 for f in forms if _wm(pn, f) or all(_wm(pn, w) for w in f.split()))
    recall = found / len(forms) if forms else 1.0
    men = set()
    for m in re.findall(r'[A-Z][A-Za-z]*(?:[ -][A-Z][A-Za-z]*)*', _sa(prediction)):
        n = _norm(m)
        if len(n) >= 3 and (' ' in n or n not in _STOP): men.add(n)
    for m in re.findall(r'[A-Za-z0-9]+(?:[./\-][A-Za-z0-9]+)*', str(prediction)):
        if any(c.isdigit() for c in m): men.add(_norm(m))
    men = {m for m in men if m}
    def ok(m):
        for f in forms:
            if m == f or _wm(f, m): return True
        if all(t in tokens for t in m.split()): return True
        d = _dg(m)
        if d and any(d in c or c in d for c in num): return True
        md = m.replace(' ', '')
        return len(md) >= 3 and any(md in x or x in md for x in fd)
    hall = sorted(m for m in men if not ok(m))
    corr = [m for m in hall if max((SequenceMatcher(None, m, f).ratio() for f in forms), default=0) >= 0.55]
    return {'halluc': len(hall)/len(men) if men else 0.0, 'recall': recall,
            'hallucinated': hall, 'corruptions': corr}
ART = {'iso': r'[A-Za-z],? \d{3,4}-\d{1,2}-\d{1,2}|\d{1,2}(st|nd|rd|th)? [A-Za-z]+ \d{3,4}-\d{1,2}-\d{1,2}',
       'paren': r'\((The [^)]+album|[0-9]{4} film|film|band|song|album|actor[^)]*|footballer[^)]*|musician[^)]*)\)',
       'unit': r'\d[\d.,]*\s*\((milli|centi|kilo)?(metres|meters|grams|litres|liters|inches)\)'}
def artrow(p): return any(re.search(x, str(p)) for x in ART.values())
def corpus_bleu_lc(preds, refs_list):
    maxr = max(len(r) for r in refs_list)
    streams = [[r[k] if k < len(r) else r[0] for r in refs_list] for k in range(maxr)]
    return sacrebleu.corpus_bleu(preds, streams, lowercase=True, tokenize='13a').score

class TNode:
    __slots__ = ('ch', 'score', 'mx', 'nterm', 'terminal')
    def __init__(self):
        self.ch = {}; self.score = None; self.mx = -1e9; self.nterm = 0; self.terminal = False
_LIT = [r'^[\d\s.,:/\-+%°"]*$', r'^\d{3,4}-\d{1,2}-\d{1,2}', r'^\d+(\.\d+)?$']
def is_literal(name):
    n = str(name).strip().strip('"').strip("'").strip()
    return len(n) < 2 or any(re.match(p, n) for p in _LIT)
def clean_surface(name):
    s = str(name).strip().strip('"').strip("'").replace('_', ' ')
    s = re.sub(r'\s*\([^)]*\)', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()
class TrieMapV2:
    def __init__(self, entity_names, tokenizer):
        self.root = TNode(); self.kept = {}
        for ni, name in enumerate(entity_names):
            if is_literal(name): continue
            base = clean_surface(name)
            if not base or is_literal(base): continue
            self.kept[ni] = base
            for v in (base, ' ' + base):
                ids = tokenizer.encode(v, add_special_tokens=False)
                if not ids: continue
                n = self.root
                for tid in ids: n = n.ch.setdefault(int(tid), TNode())
                n.terminal = True
        self._cache(self.root)
    def _cache(self, n):
        nt = 1 if n.terminal else 0
        for c in n.ch.values():
            self._cache(c); nt += c.nterm
        n.nterm = nt
    def suffix_matches(self, gen_ids, lookback=20):
        out = []
        for start in range(max(0, len(gen_ids) - lookback), len(gen_ids)):
            n = self.root; okk = True
            for pos in range(start, len(gen_ids)):
                t = int(gen_ids[pos])
                if t not in n.ch: okk = False; break
                n = n.ch[t]
            if okk and n is not self.root: out.append((len(gen_ids) - start, n))
        return out
def hard_mask_v2(logits, trie, gen_ids):
    best = None
    for depth, node in trie.suffix_matches(gen_ids):
        if node.terminal or not node.ch: continue
        if (depth >= LOCK_MIN_DEPTH or node.nterm == 1) and (best is None or depth > best[0]):
            best = (depth, node)
    if best is None: return logits, False
    legal = list(best[1].ch.keys())
    if float(logits.max()) - max(float(logits[t]) for t in legal) > ESCAPE_MARGIN:
        return logits, False
    m = torch.full_like(logits, -1e9); m[legal] = 0.0
    return logits + m, True
print('Development-only metric and Hard-v2 utilities ready.')


In [ ]:
# 5. Selector module + gold span labels from references.
class EntitySelector(nn.Module):
    # scores {NONE} ∪ {entities} from [h_t ; e_i ; cov_i]
    def __init__(self, d=768, hid=256):
        super().__init__()
        self.ent_mlp = nn.Sequential(nn.Linear(2*d + 1, hid), nn.ReLU(), nn.Linear(hid, 1))
        self.none_mlp = nn.Sequential(nn.Linear(d, hid), nn.ReLU(), nn.Linear(hid, 1))
    def forward(self, h, ents, cov):
        # h: (L,d) | ents: (N,d) | cov: (L,N) in {0,1}
        L, d = h.shape; N = ents.shape[0]
        he = torch.cat([h.unsqueeze(1).expand(L, N, d), ents.unsqueeze(0).expand(L, N, d),
                        cov.unsqueeze(-1)], dim=-1)
        s_ent = self.ent_mlp(he).squeeze(-1)          # (L,N)
        s_none = self.none_mlp(h)                     # (L,1)
        return torch.cat([s_none, s_ent], dim=-1)     # (L, 1+N); class 0 = NONE

def target_labels(ex, graph, target_ids):
    # label per decoder-state position t (predicting token t of target_ids):
    #  0 = NONE, i+1 = entity i STARTS at t, -100 = inside a span (ignored)
    names = list(getattr(graph, 'entity_names', []) or [])
    surfaces = [clean_surface(n) if not is_literal(n) else '' for n in names]
    lab = np.zeros(len(target_ids), dtype=np.int64)
    text = ex['target']
    spans = find_entity_token_spans(text, [s if s else '§none§' for s in surfaces], tokenizer)
    for ni, (s, e) in enumerate(spans):
        if s < 0 or not surfaces[ni]: continue
        if s < len(lab): lab[s] = ni + 1
        for k in range(s + 1, min(e, len(lab))): lab[k] = -100
    return lab
print('selector defined')


In [ ]:
from torch_geometric.nn import RGCNConv

class ResidualConstraintRGCNHead(nn.Module):
    """One relation-aware constraint head with function-preserving initialization."""
    def __init__(self, d=768, bottleneck=256, num_relations=None, num_bases=8, dropout=0.1):
        super().__init__()
        assert num_relations is not None
        self.conv = RGCNConv(d, bottleneck, num_relations, num_bases=min(num_bases, num_relations))
        self.norm = nn.LayerNorm(bottleneck)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(bottleneck, d)
        nn.init.zeros_(self.out.weight)
        nn.init.zeros_(self.out.bias)
        self.residual_scale = nn.Parameter(torch.tensor(1.0))

    def forward(self, g_fusion, edge_index, edge_type):
        h = self.conv(g_fusion, edge_index, edge_type)
        h = self.dropout(self.norm(F.gelu(h)))
        delta = self.out(h)
        g_constraint = g_fusion + self.residual_scale * delta
        return g_constraint, delta

class DualHeadAdaptiveGNN(nn.Module):
    def __init__(self, d, bottleneck, num_relations, num_bases, dropout):
        super().__init__()
        self.constraint_head = ResidualConstraintRGCNHead(
            d=d, bottleneck=bottleneck, num_relations=num_relations,
            num_bases=num_bases, dropout=dropout,
        )
        self.selector = EntitySelector(d=d, hid=256)

    def constraint_states(self, g_fusion, edge_index, edge_type):
        return self.constraint_head(g_fusion, edge_index, edge_type)

def nparams(module, trainable=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable))

print('Dual-head architecture ready.')


In [ ]:
# 8. Load the frozen shared selector and initialize the residual dual model.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_json_dump(value, path, indent=2):
    temporary = path + '.tmp'
    with open(temporary, 'w', encoding='utf-8') as f:
        json.dump(value, f, indent=indent, ensure_ascii=False)
    os.replace(temporary, path)

def atomic_torch_save(value, path):
    temporary = path + '.tmp'
    torch.save(value, temporary)
    os.replace(temporary, path)

current_ckpt = torch.load(CURRENT_SELECTOR_CKPT, map_location=DEVICE)
current_selector = EntitySelector(d=MODEL_DIM, hid=256).to(DEVICE)
current_selector.load_state_dict(current_ckpt['state'])
current_selector.eval()
for parameter in current_selector.parameters():
    parameter.requires_grad_(False)

def build_fresh_dual_model():
    model = DualHeadAdaptiveGNN(
        d=MODEL_DIM,
        bottleneck=CONSTRAINT_BOTTLENECK,
        num_relations=num_relations,
        num_bases=CONSTRAINT_NUM_BASES,
        dropout=CONSTRAINT_DROPOUT,
    ).to(DEVICE)
    model.selector.load_state_dict(current_ckpt['state'])
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    for parameter in model.constraint_head.parameters():
        parameter.requires_grad_(True)
    for parameter in model.selector.ent_mlp.parameters():
        parameter.requires_grad_(True)
    for parameter in model.selector.none_mlp.parameters():
        parameter.requires_grad_(TRAIN_NONE_MLP)
    return model

dual_model = build_fresh_dual_model()
print('Shared selector SHA256:', sha256_file(CURRENT_SELECTOR_CKPT))
print('Epoch-0 residual output is zero:', float(dual_model.constraint_head.out.weight.abs().max().detach().cpu()))


In [ ]:
def rebuild_v5_splits():
    universe = list(range(len(data['dev'])))
    assert len(universe) == 1667
    a = set(random.Random(7).sample(universe, 300))
    pool = [i for i in universe if i not in a]
    b = set(random.Random(21).sample(pool, 500))
    pool = [i for i in universe if i not in a and i not in b]
    c = set(random.Random(33).sample(pool, 300))
    unused = [i for i in universe if i not in a and i not in b and i not in c]
    val = set(random.Random(55).sample(unused, 167))
    remaining = [i for i in unused if i not in val]
    tune = set(random.Random(66).sample(remaining, 200))
    confirm = set(remaining) - tune
    groups = {
        'excluded_historical_seed7': sorted(a),
        'excluded_historical_seed21': sorted(b),
        'excluded_inspected_v4_tune': sorted(c),
        'selector_validation': sorted(val),
        'generation_tune': sorted(tune),
        'generation_confirm': sorted(confirm),
    }
    raw = json.dumps(groups, sort_keys=True, separators=(',', ':')).encode()
    return {'version': 'v5_operating_point_match', 'dev_size': 1667, 'groups': groups,
            'sha256': hashlib.sha256(raw).hexdigest()}

if os.path.exists(CURRENT_SPLIT_MANIFEST):
    SPLITS = json.load(open(CURRENT_SPLIT_MANIFEST, encoding='utf-8'))
else:
    SPLITS = rebuild_v5_splits()
SELECTOR_VAL_IDX = SPLITS['groups']['selector_validation']
GEN_TUNE_IDX = SPLITS['groups']['generation_tune']
GEN_CONFIRM_IDX = SPLITS['groups']['generation_confirm']
assert len(SELECTOR_VAL_IDX) == 167 and len(GEN_TUNE_IDX) == 200 and len(GEN_CONFIRM_IDX) == 200
assert not set(SELECTOR_VAL_IDX) & set(GEN_TUNE_IDX)
assert not set(SELECTOR_VAL_IDX) & set(GEN_CONFIRM_IDX)
assert not set(GEN_TUNE_IDX) & set(GEN_CONFIRM_IDX)
atomic_json_dump(SPLITS, os.path.join(OUTPUT_DIR, 'dual_head_split_manifest.json'))
print({k: len(SPLITS['groups'][k]) for k in ('selector_validation','generation_tune','generation_confirm')})


In [ ]:
from torch_geometric.data import Batch as PyGBatch
pad_id = tokenizer.pad_token_id
dec_start = fusion_model.bart.config.decoder_start_token_id

@torch.no_grad()
def teacher_features(examples, graph_list):
    enc_batch = tokenizer(
        [x['linearized'] for x in examples], max_length=MAX_INPUT_LEN,
        truncation=True, padding=True, return_tensors='pt',
    ).to(DEVICE)
    tgt_batch = tokenizer(
        text_target=[x['target'] for x in examples], max_length=MAX_TARGET_LEN,
        truncation=True, padding=True, return_tensors='pt',
    )
    target_ids = tgt_batch['input_ids']
    decoder_ids = torch.cat([
        torch.full((target_ids.size(0), 1), dec_start, dtype=target_ids.dtype),
        target_ids[:, :-1],
    ], dim=1).to(DEVICE)
    gb = PyGBatch.from_data_list(graph_list).to(DEVICE)
    enc = fusion_model.bart.model.encoder(
        input_ids=enc_batch['input_ids'], attention_mask=enc_batch['attention_mask'])
    g_fusion, _ = fusion_model.rgcn(gb.x, gb.edge_index, gb.edge_type)
    g_pad, g_mask = pad_kg_nodes(g_fusion, gb.batch, len(examples))
    dec = fusion_model.bart.model.decoder(
        input_ids=decoder_ids, encoder_hidden_states=enc.last_hidden_state,
        encoder_attention_mask=enc_batch['attention_mask'])
    h_fused, _, _ = fusion_model.kg_cross_attention(dec.last_hidden_state, g_pad, g_mask)
    return {
        'h_fused': h_fused.detach(), 'g_fusion': g_fusion.detach(),
        'batch': gb.batch.detach(), 'edge_index': gb.edge_index.detach(),
        'edge_type': gb.edge_type.detach(), 'target_ids': target_ids,
    }

def coverage_from_labels(labels, n_entities):
    cov = torch.zeros(len(labels), n_entities, device=DEVICE)
    seen = set()
    for t in range(len(labels)):
        for i in seen: cov[t, i] = 1.0
        y = int(labels[t].item())
        if y > 0: seen.add(y - 1)
    return cov

def example_pack(ex, graph, target_row, h_row, entity_states):
    names = list(getattr(graph, 'entity_names', []) or [])
    if not names: return None
    assert entity_states.size(0) == len(names), (entity_states.size(0), len(names))
    L = int((target_row != pad_id).sum().item())
    if L < 2: return None
    labels = torch.from_numpy(target_labels(ex, graph, target_row[:L].tolist())).long().to(DEVICE)
    return {
        'h': h_row[:L].float(), 'ents': entity_states.float(),
        'labels': labels, 'cov': coverage_from_labels(labels, len(names)), 'L': L,
    }


In [ ]:
# 11. Intrinsic evaluation and threshold-aware calibration records.
def selector_ce(logits, labels):
    weights = torch.ones(logits.size(-1), device=logits.device)
    weights[0] = NONE_WEIGHT
    return F.cross_entropy(logits, labels, weight=weights, ignore_index=-100)

@torch.no_grad()
def evaluate_argmax(model_key, indices, batch_size=VAL_BATCH_SIZE, verbose=True):
    assert model_key in ('shared','dual')
    fusion_model.eval(); current_selector.eval(); dual_model.eval()
    tp = fp = fn = exact = true_starts = none_ok = none_n = 0
    loss_sum = 0.0; loss_n = 0; delta_sum = 0.0; delta_n = 0
    for start in range(0, len(indices), batch_size):
        ids = indices[start:start + batch_size]
        examples = [data['dev'][i] for i in ids]
        graph_list = [graphs['dev'][i] for i in ids]
        features = teacher_features(examples, graph_list)
        if model_key == 'dual':
            all_entities, delta = dual_model.constraint_states(
                features['g_fusion'].float(), features['edge_index'], features['edge_type'])
            delta_sum += float(delta.norm(dim=-1).sum().cpu()); delta_n += delta.size(0)
            selector = dual_model.selector
        else:
            all_entities = features['g_fusion'].float()
            selector = current_selector
        for j, (example, graph) in enumerate(zip(examples, graph_list)):
            node_indices = (features['batch'] == j).nonzero(as_tuple=False).flatten()
            pack = example_pack(
                example, graph, features['target_ids'][j], features['h_fused'][j],
                all_entities[node_indices],
            )
            if pack is None:
                continue
            logits = selector(pack['h'], pack['ents'], pack['cov'])
            loss = selector_ce(logits, pack['labels'])
            if torch.isfinite(loss):
                loss_sum += float(loss.cpu()); loss_n += 1
            predictions = logits.argmax(-1).cpu()
            gold = pack['labels'].cpu()
            for t in range(pack['L']):
                y, pred = int(gold[t]), int(predictions[t])
                if y == -100:
                    continue
                if y > 0:
                    true_starts += 1
                    if pred > 0:
                        tp += 1
                        exact += int(pred == y)
                    else:
                        fn += 1
                else:
                    none_n += 1
                    if pred > 0:
                        fp += 1
                    else:
                        none_ok += 1
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    result = {
        'model_key': model_key,
        'loss': loss_sum / max(loss_n, 1),
        'P': precision, 'R': recall, 'F1': f1,
        'identity_accuracy_among_detected_starts': exact / max(tp, 1),
        'NONE_accuracy': none_ok / max(none_n, 1),
        'true_starts': true_starts, 'detected_gold_starts': tp,
        'false_triggers': fp, 'missed_starts': fn,
        'exact_entity_identity': exact,
        'mean_delta_norm': delta_sum / max(delta_n, 1) if model_key == 'dual' else 0.0,
    }
    if verbose:
        print(model_key, result)
    return result

@torch.no_grad()
def collect_calibration_records(model_key, indices, batch_size=VAL_BATCH_SIZE):
    assert model_key in ('shared','dual')
    fusion_model.eval(); current_selector.eval(); dual_model.eval()
    records = []
    for start in range(0, len(indices), batch_size):
        ids = indices[start:start + batch_size]
        examples = [data['dev'][i] for i in ids]
        graph_list = [graphs['dev'][i] for i in ids]
        features = teacher_features(examples, graph_list)
        if model_key == 'dual':
            all_entities, _ = dual_model.constraint_states(
                features['g_fusion'].float(), features['edge_index'], features['edge_type'])
            selector = dual_model.selector
        else:
            all_entities = features['g_fusion'].float()
            selector = current_selector
        for j, (example, graph, dev_index) in enumerate(zip(examples, graph_list, ids)):
            node_indices = (features['batch'] == j).nonzero(as_tuple=False).flatten()
            pack = example_pack(
                example, graph, features['target_ids'][j], features['h_fused'][j],
                all_entities[node_indices],
            )
            if pack is None:
                continue
            logits = selector(pack['h'], pack['ents'], pack['cov'])
            probabilities = torch.softmax(logits, dim=-1)
            best_entity_prob, best_entity_offset = probabilities[:, 1:].max(dim=-1)
            gold = pack['labels']
            for position in range(pack['L']):
                y = int(gold[position])
                if y == -100:
                    continue
                records.append({
                    'dev_index': int(dev_index),
                    'position': int(position),
                    'gold_class': y,
                    'gold_is_start': y > 0,
                    'p_none': float(probabilities[position, 0].cpu()),
                    'best_entity_probability': float(best_entity_prob[position].cpu()),
                    'best_entity_class': int(best_entity_offset[position].cpu()) + 1,
                })
    return pd.DataFrame(records)

def threshold_metrics(records, tau):
    frame = records.copy()
    frame['trigger'] = frame['best_entity_probability'] >= tau
    frame['correct_identity'] = frame['trigger'] & frame['gold_is_start'] & (frame['best_entity_class'] == frame['gold_class'])
    tp = int((frame['trigger'] & frame['gold_is_start']).sum())
    fp = int((frame['trigger'] & ~frame['gold_is_start']).sum())
    fn = int((~frame['trigger'] & frame['gold_is_start']).sum())
    exact = int(frame['correct_identity'].sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    return {
        'tau': float(tau),
        'P': precision,
        'R': recall,
        'F1': 2 * precision * recall / max(precision + recall, 1e-12),
        'identity_among_detected_starts': exact / max(tp, 1),
        'predicted_triggers': int(frame['trigger'].sum()),
        'false_triggers': fp,
        'missed_starts': fn,
    }


In [ ]:
# 12. Reproduce two dual epochs and save each checkpoint. No F1-based checkpoint selection.
CHECKPOINT_DIR = f'{OUTPUT_DIR}/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
TRAIN_STATE_PATH = f'{CHECKPOINT_DIR}/dual_training_state.pt'
HISTORY_PATH = f'{OUTPUT_DIR}/dual_two_epoch_history.json'

if FORCE_RETRAIN:
    for path in glob.glob(f'{CHECKPOINT_DIR}/dual_epoch_*.pt') + [TRAIN_STATE_PATH, HISTORY_PATH]:
        if os.path.exists(path):
            os.remove(path)
    dual_model = build_fresh_dual_model()

optimizer = torch.optim.AdamW([
    {
        'params': [p for p in dual_model.constraint_head.parameters() if p.requires_grad],
        'lr': HEAD_LR, 'weight_decay': WEIGHT_DECAY,
    },
    {
        'params': [p for p in dual_model.selector.ent_mlp.parameters() if p.requires_grad],
        'lr': SELECTOR_ENTITY_LR, 'weight_decay': WEIGHT_DECAY,
    },
])

start_epoch = 0
history = []
if os.path.isfile(TRAIN_STATE_PATH) and not FORCE_RETRAIN:
    resume = torch.load(TRAIN_STATE_PATH, map_location=DEVICE)
    dual_model.load_state_dict(resume['state'])
    optimizer.load_state_dict(resume['optimizer'])
    start_epoch = int(resume['epoch'])
    history = list(resume.get('history', []))
    print('Resuming after epoch', start_epoch)

train_indices = list(range(len(data['train'])))
for epoch in range(start_epoch, DUAL_EPOCHS):
    dual_model.train(); fusion_model.eval()
    random.Random(SEED + epoch).shuffle(train_indices)
    ce_sum = reg_sum = total_sum = 0.0
    updates = 0
    start_time = time.time()
    for batch_start in range(0, len(train_indices), TRAIN_BATCH_SIZE):
        indices = train_indices[batch_start:batch_start + TRAIN_BATCH_SIZE]
        examples = [data['train'][i] for i in indices]
        graph_list = [graphs['train'][i] for i in indices]
        features = teacher_features(examples, graph_list)
        constraint_entities, delta = dual_model.constraint_states(
            features['g_fusion'].float(), features['edge_index'], features['edge_type'])
        ce = None
        usable = 0
        for j, (example, graph) in enumerate(zip(examples, graph_list)):
            node_indices = (features['batch'] == j).nonzero(as_tuple=False).flatten()
            pack = example_pack(
                example, graph, features['target_ids'][j], features['h_fused'][j],
                constraint_entities[node_indices],
            )
            if pack is None:
                continue
            loss_i = selector_ce(
                dual_model.selector(pack['h'], pack['ents'], pack['cov']),
                pack['labels'],
            )
            if torch.isfinite(loss_i):
                ce = loss_i if ce is None else ce + loss_i
                usable += 1
        if usable == 0:
            continue
        ce = ce / usable
        regularization = delta.pow(2).mean()
        total = ce + DELTA_L2_WEIGHT * regularization
        optimizer.zero_grad(set_to_none=True)
        total.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in dual_model.parameters() if p.requires_grad], GRAD_CLIP)
        optimizer.step()

        ce_sum += float(ce.detach().cpu())
        reg_sum += float(regularization.detach().cpu())
        total_sum += float(total.detach().cpu())
        updates += 1
        if updates % 50 == 0:
            print(
                f'epoch {epoch + 1} step {updates}: CE={ce_sum / updates:.4f} '
                f'scale={float(dual_model.constraint_head.residual_scale.detach().cpu()):.4f} '
                f'({time.time() - start_time:.0f}s)'
            )

    validation = evaluate_argmax('dual', SELECTOR_VAL_IDX)
    checkpoint_path = f'{CHECKPOINT_DIR}/dual_epoch_{epoch + 1}.pt'
    checkpoint = {
        'state': dual_model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'epoch': epoch + 1,
        'validation': validation,
        'config': {
            'head_lr': HEAD_LR,
            'selector_entity_lr': SELECTOR_ENTITY_LR,
            'none_weight': NONE_WEIGHT,
            'delta_l2_weight': DELTA_L2_WEIGHT,
            'seed': SEED,
        },
    }
    atomic_torch_save(checkpoint, checkpoint_path)
    history.append({
        'epoch': epoch + 1,
        'mean_ce_loss': ce_sum / max(updates, 1),
        'mean_regularization': reg_sum / max(updates, 1),
        'mean_total_loss': total_sum / max(updates, 1),
        'updates': updates,
        'elapsed_sec': time.time() - start_time,
        'validation': validation,
        'checkpoint': checkpoint_path,
        'sha256': sha256_file(checkpoint_path),
    })
    atomic_json_dump(history, HISTORY_PATH)
    atomic_torch_save({
        'state': dual_model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'epoch': epoch + 1,
        'history': history,
    }, TRAIN_STATE_PATH)
    print('Saved', checkpoint_path)

EPOCH_PATHS = {
    'dual_e1': f'{CHECKPOINT_DIR}/dual_epoch_1.pt',
    'dual_e2': f'{CHECKPOINT_DIR}/dual_epoch_2.pt',
}
for key, path in EPOCH_PATHS.items():
    assert os.path.isfile(path), f'Missing {key}: {path}'
print({key: sha256_file(path) for key, path in EPOCH_PATHS.items()})


In [ ]:
# 13. Verify epoch metrics and create threshold-aware intrinsic calibration curves.
import pandas as pd
from IPython.display import display

shared_argmax = evaluate_argmax('shared', SELECTOR_VAL_IDX)
argmax_rows = [{**shared_argmax, 'checkpoint': CURRENT_SELECTOR_CKPT}]
calibration_frames = {}

calibration_frames['shared'] = collect_calibration_records('shared', SELECTOR_VAL_IDX)

for model_key, checkpoint_path in EPOCH_PATHS.items():
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    dual_model.load_state_dict(checkpoint['state'])
    dual_model.eval()
    metrics = evaluate_argmax('dual', SELECTOR_VAL_IDX)
    metrics['model_key'] = model_key
    metrics['checkpoint'] = checkpoint_path
    argmax_rows.append(metrics)
    calibration_frames[model_key] = collect_calibration_records('dual', SELECTOR_VAL_IDX)

argmax_df = pd.DataFrame(argmax_rows)
display(argmax_df)
argmax_df.to_csv(f'{OUTPUT_DIR}/intrinsic_argmax_by_checkpoint.csv', index=False)

threshold_rows = []
for model_key, records in calibration_frames.items():
    for tau in TAU_GRID:
        threshold_rows.append({'model_key': model_key, **threshold_metrics(records, tau)})
intrinsic_threshold_df = pd.DataFrame(threshold_rows)
display(intrinsic_threshold_df)
intrinsic_threshold_df.to_csv(f'{OUTPUT_DIR}/intrinsic_threshold_sweep.csv', index=False)

# Explicitly surface the known epoch-2 recall question.
print(intrinsic_threshold_df[intrinsic_threshold_df['model_key'].isin(['dual_e1','dual_e2'])]
      .sort_values(['model_key','tau'])[['model_key','tau','P','R','F1','predicted_triggers']])


In [ ]:
# 14. Adaptive decoder parameterized by checkpoint and threshold.
ACTIVE_MODEL_KEY = None

def activate_model(model_key):
    global ACTIVE_MODEL_KEY
    if model_key == 'shared':
        current_selector.eval()
    else:
        checkpoint_path = EPOCH_PATHS[model_key]
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        dual_model.load_state_dict(checkpoint['state'])
        dual_model.eval()
        for parameter in dual_model.parameters():
            parameter.requires_grad_(False)
    ACTIVE_MODEL_KEY = model_key

@torch.no_grad()
def decode_adaptive(ex, graph, model_key, tau, use_hard=False, return_events=False):
    assert model_key in ('shared','dual_e1','dual_e2')
    if ACTIVE_MODEL_KEY != model_key:
        activate_model(model_key)

    encoded = tokenizer(
        ex['linearized'], max_length=MAX_INPUT_LEN, truncation=True,
        padding='max_length', return_tensors='pt')
    input_ids = encoded['input_ids'].to(DEVICE)
    attention_mask = encoded['attention_mask'].to(DEVICE)
    graph_batch = torch.zeros(graph.x.size(0), dtype=torch.long, device=DEVICE)

    encoder_output = fusion_model.bart.model.encoder(
        input_ids=input_ids, attention_mask=attention_mask)
    edge_index = graph.edge_index.to(DEVICE)
    edge_type = graph.edge_type.to(DEVICE)
    g_fusion, _ = fusion_model.rgcn(graph.x.to(DEVICE), edge_index, edge_type)
    graph_padded, graph_mask = pad_kg_nodes(g_fusion, graph_batch, 1)

    if model_key == 'shared':
        selector_entities = g_fusion.float()
        selector = current_selector
    else:
        selector_entities, _ = dual_model.constraint_states(
            g_fusion.float(), edge_index, edge_type)
        selector = dual_model.selector

    names = list(getattr(graph, 'entity_names', []) or [])
    trie = TrieMapV2(names, tokenizer)
    entity_token_ids = {}
    for entity_index, surface in trie.kept.items():
        beginning = tokenizer.encode(surface, add_special_tokens=False)
        continuation = tokenizer.encode(' ' + surface, add_special_tokens=False)
        if beginning and continuation:
            entity_token_ids[entity_index] = (beginning, continuation)

    start_id = fusion_model.bart.config.decoder_start_token_id
    eos_id = fusion_model.bart.config.eos_token_id
    generated = [start_id]
    past = None
    committed = None
    selector_triggers = 0
    hard_triggers = 0
    events = []

    for step in range(MAX_GEN_LEN - 1):
        decoder_input = torch.tensor([[generated[-1]]], dtype=torch.long, device=DEVICE)
        decoder_output = fusion_model.bart.model.decoder(
            input_ids=decoder_input,
            encoder_hidden_states=encoder_output.last_hidden_state,
            encoder_attention_mask=attention_mask,
            past_key_values=past,
            use_cache=True,
        )
        past = decoder_output.past_key_values
        h_fused, _, _ = fusion_model.kg_cross_attention(
            decoder_output.last_hidden_state, graph_padded, graph_mask)
        logits = add_final_logits_bias(
            fusion_model.bart,
            fusion_model.bart.lm_head(h_fused),
        )[:, -1, :].squeeze(0).float().cpu()

        if committed is not None:
            next_token = committed['ids'][committed['position']]
            committed['position'] += 1
            if committed['position'] >= len(committed['ids']):
                committed = None
        else:
            if use_hard:
                constrained_logits, active = hard_mask_v2(logits, trie, generated[1:])
                hard_triggers += int(active)
            else:
                constrained_logits = logits
            next_token = int(constrained_logits.argmax())

            if entity_token_ids:
                decoded = tokenizer.decode(generated[1:], skip_special_tokens=True)
                decoded_norm = _norm(decoded)
                coverage_flags = [
                    1.0 if (trie.kept.get(i) and _wm(decoded_norm, _norm(trie.kept[i]))) else 0.0
                    for i in range(len(names))
                ]
                coverage = torch.tensor([coverage_flags], device=DEVICE)
                selector_logits = selector(
                    h_fused[:, -1, :].float(), selector_entities.float(), coverage
                ).squeeze(0)
                entity_logits = selector_logits[1:].clone()
                for entity_index in range(len(names)):
                    if entity_index not in entity_token_ids or coverage_flags[entity_index] > 0:
                        entity_logits[entity_index] = -1e9
                probabilities = torch.softmax(torch.cat([selector_logits[:1], entity_logits]), dim=-1)
                if probabilities.numel() > 1:
                    best_entity = int(probabilities[1:].argmax())
                    probability = float(probabilities[1 + best_entity])
                    if probability >= tau and best_entity in entity_token_ids:
                        sequence = entity_token_ids[best_entity][0] if len(generated) == 1 else entity_token_ids[best_entity][1]
                        gap = float(logits.max()) - float(logits[sequence[0]])
                        if gap <= COMPATIBILITY_MARGIN:
                            next_token = sequence[0]
                            selector_triggers += 1
                            if len(sequence) > 1:
                                committed = {'ids': sequence, 'position': 1}
                            events.append({
                                'step': step,
                                'entity_index': best_entity,
                                'entity': trie.kept.get(best_entity, names[best_entity]),
                                'probability': probability,
                                'bart_first_token_gap': gap,
                            })
        if next_token == eos_id:
            break
        generated.append(next_token)

    result = {
        'prediction': tokenizer.decode(generated[1:], skip_special_tokens=True),
        'selector_triggers': selector_triggers,
        'hard_v2_triggers': hard_triggers,
    }
    if return_events:
        result['events'] = events
    return result

print('Checkpoint- and threshold-parameterized decoder ready.')


In [ ]:
# 15. Development items, resume-safe generation, metrics, and trigger accounting.
def dev_items(indices):
    items = []
    for dev_index in indices:
        example = dict(data['dev'][dev_index])
        references = example.get('all_targets') or [example.get('target', '')]
        example['all_targets'] = [str(x).strip() for x in references if str(x).strip()]
        items.append({'idx': dev_index, 'ex': example, 'graph': graphs['dev'][dev_index]})
    return items

def tau_tag(tau):
    return f'{tau:.2f}'.replace('.', 'p')

def model_fingerprint(model_key):
    if model_key == 'shared':
        return sha256_file(CURRENT_SELECTOR_CKPT)[:12]
    return sha256_file(EPOCH_PATHS[model_key])[:12]

def run_cached(items, split_name, model_key, tau, use_hard=False, save_events=False):
    hard_tag = 'hard' if use_hard else 'plain'
    fingerprint = model_fingerprint(model_key)
    stem = f'{split_name}_{model_key}_{fingerprint}_tau{tau_tag(tau)}_{hard_tag}'
    prediction_path = f'{OUTPUT_DIR}/preds_{stem}.json'
    trigger_path = f'{OUTPUT_DIR}/triggers_{stem}.json'
    event_path = f'{OUTPUT_DIR}/events_{stem}.json'

    predictions = json.load(open(prediction_path)) if os.path.isfile(prediction_path) else []
    triggers = json.load(open(trigger_path)) if os.path.isfile(trigger_path) else []
    events = json.load(open(event_path)) if save_events and os.path.isfile(event_path) else []
    assert len(predictions) == len(triggers) <= len(items)
    if save_events:
        assert len(events) == len(predictions)

    activate_model(model_key)
    start_time = time.time()
    for index in range(len(predictions), len(items)):
        result = decode_adaptive(
            items[index]['ex'], items[index]['graph'], model_key=model_key,
            tau=tau, use_hard=use_hard, return_events=save_events)
        predictions.append(result['prediction'])
        triggers.append({
            'selector': int(result['selector_triggers']),
            'hard_v2': int(result['hard_v2_triggers']),
        })
        if save_events:
            events.append(result.get('events', []))
        if (index + 1) % 25 == 0 or index + 1 == len(items):
            atomic_json_dump(predictions, prediction_path)
            atomic_json_dump(triggers, trigger_path)
            if save_events:
                atomic_json_dump(events, event_path)
        if (index + 1) % 100 == 0:
            print(f'[{stem}] {index + 1}/{len(items)} ({time.time() - start_time:.0f}s)')
    return predictions, triggers

def score_block(predictions, items, triggers):
    scores = [grounding_score(prediction, item['ex']['triples']) for prediction, item in zip(predictions, items)]
    selector_counts = np.array([int(row.get('selector', 0)) for row in triggers], dtype=int)
    hard_counts = np.array([int(row.get('hard_v2', 0)) for row in triggers], dtype=int)
    return {
        'n': len(items),
        'bleu': corpus_bleu_lc(predictions, [item['ex']['all_targets'] for item in items]),
        'halluc': float(np.mean([score['halluc'] for score in scores])),
        'recall': float(np.mean([score['recall'] for score in scores])),
        'corr_rows': int(sum(len(score['corruptions']) > 0 for score in scores)),
        'art_rows': int(sum(artrow(prediction) for prediction in predictions)),
        'total_selector_triggers': int(selector_counts.sum()),
        'examples_with_selector_trigger': int((selector_counts > 0).sum()),
        'mean_selector_triggers': float(selector_counts.mean()),
        'total_hard_v2_triggers': int(hard_counts.sum()),
    }


In [ ]:
# 16. Tune-split threshold sweep for shared, dual epoch 1, and dual epoch 2.
TUNE_ITEMS = dev_items(GEN_TUNE_IDX)
MODEL_KEYS = ['shared','dual_e1','dual_e2']
TUNE_PREDS = {}
TUNE_TRIGGERS = {}
tune_rows = []

if RUN_TUNE_SWEEP:
    for model_key in MODEL_KEYS:
        for tau in TAU_GRID:
            cache_key = f'{model_key}_tau{tau_tag(tau)}'
            predictions, triggers = run_cached(
                TUNE_ITEMS, 'tune', model_key, tau, use_hard=False, save_events=False)
            TUNE_PREDS[cache_key] = predictions
            TUNE_TRIGGERS[cache_key] = triggers
            metrics = score_block(predictions, TUNE_ITEMS, triggers)
            row = {'split': 'tune', 'model_key': model_key, 'tau': tau, **metrics}
            tune_rows.append(row)
            pd.DataFrame(tune_rows).to_csv(f'{OUTPUT_DIR}/tune_operating_point_sweep.csv', index=False)
            print(row)

    tune_df = pd.DataFrame(tune_rows).sort_values(['model_key','tau']).reset_index(drop=True)
    display(tune_df)
else:
    tune_df = pd.read_csv(f'{OUTPUT_DIR}/tune_operating_point_sweep.csv')

assert not tune_df.empty


In [ ]:
# 17. Plot generation quality against trigger volume and construct matched pairs.
import matplotlib.pyplot as plt

for metric, ylabel, filename in [
    ('recall', 'Entity recall', 'tune_recall_vs_triggers.png'),
    ('halluc', 'Entity hallucination', 'tune_halluc_vs_triggers.png'),
    ('bleu', 'BLEU', 'tune_bleu_vs_triggers.png'),
]:
    plt.figure(figsize=(8, 5))
    for model_key, group in tune_df.groupby('model_key'):
        group = group.sort_values('total_selector_triggers')
        plt.plot(group['total_selector_triggers'], group[metric], marker='o', label=model_key)
        for _, row in group.iterrows():
            plt.annotate(f"{row['tau']:.2f}", (row['total_selector_triggers'], row[metric]), fontsize=7)
    plt.xlabel('Total selector triggers on tune-200')
    plt.ylabel(ylabel)
    plt.title(f'{ylabel} versus selector trigger budget')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{filename}', dpi=160)
    plt.show()

reference_rows = tune_df[
    (tune_df['model_key'] == 'shared')
    & np.isclose(tune_df['tau'].astype(float), SHARED_REFERENCE_TAU)
]
assert len(reference_rows) == 1
reference = reference_rows.iloc[0].to_dict()
reference_budget = int(reference['total_selector_triggers'])

matched_rows = []
for shared_tau, shared_row in tune_df[tune_df['model_key'] == 'shared'].groupby('tau'):
    shared_point = shared_row.iloc[0]
    for dual_key in ('dual_e1','dual_e2'):
        candidates = tune_df[tune_df['model_key'] == dual_key].copy()
        candidates['trigger_distance'] = (candidates['total_selector_triggers'] - shared_point['total_selector_triggers']).abs()
        candidate = candidates.sort_values(['trigger_distance','tau']).iloc[0]
        matched_rows.append({
            'shared_tau': float(shared_tau),
            'shared_triggers': int(shared_point['total_selector_triggers']),
            'shared_recall': float(shared_point['recall']),
            'shared_halluc': float(shared_point['halluc']),
            'shared_bleu': float(shared_point['bleu']),
            'dual_model_key': dual_key,
            'dual_tau': float(candidate['tau']),
            'dual_triggers': int(candidate['total_selector_triggers']),
            'relative_trigger_difference': float(
                abs(candidate['total_selector_triggers'] - shared_point['total_selector_triggers'])
                / max(shared_point['total_selector_triggers'], 1)
            ),
            'recall_delta_pp': 100 * float(candidate['recall'] - shared_point['recall']),
            'halluc_delta_pp': 100 * float(candidate['halluc'] - shared_point['halluc']),
            'bleu_delta': float(candidate['bleu'] - shared_point['bleu']),
        })
matched_df = pd.DataFrame(matched_rows)
matched_df.to_csv(f'{OUTPUT_DIR}/tune_matched_frontier.csv', index=False)
display(matched_df)


In [ ]:
# 18. Select one dual operating point using tune data only.
dual_candidates = tune_df[tune_df['model_key'].isin(['dual_e1','dual_e2'])].copy()
dual_candidates['trigger_distance'] = (dual_candidates['total_selector_triggers'] - reference_budget).abs()
dual_candidates['relative_trigger_difference'] = dual_candidates['trigger_distance'] / max(reference_budget, 1)
eligible = dual_candidates[dual_candidates['relative_trigger_difference'] <= MATCH_TOLERANCE_REL].copy()

selection_rule = {
    'reference': {'model_key': 'shared', 'tau': SHARED_REFERENCE_TAU, 'trigger_budget': reference_budget},
    'matching_tolerance_relative': MATCH_TOLERANCE_REL,
    'candidate_checkpoints': ['dual_e1','dual_e2'],
    'selection': 'closest total trigger budget; tune recall, hallucination, and BLEU are tie-breakers only',
}

if len(eligible):
    # Primary rule: match trigger budget as closely as the discrete threshold grid permits.
    # Generation recall is used only as a tie-breaker between equally close points.
    selected = eligible.sort_values(
        ['trigger_distance','recall','halluc','bleu'],
        ascending=[True, False, True, False],
    ).iloc[0]
    selection_reason = 'closest_trigger_budget_within_tolerance'
else:
    selected = dual_candidates.sort_values(
        ['trigger_distance','recall','halluc','bleu'],
        ascending=[True, False, True, False],
    ).iloc[0]
    selection_reason = 'no_candidate_within_tolerance; globally_closest_trigger_budget'

SELECTED_DUAL_KEY = str(selected['model_key'])
SELECTED_DUAL_TAU = float(selected['tau'])
selected_dict = {
    key: (value.item() if isinstance(value, np.generic) else value)
    for key, value in selected.to_dict().items()
}
selection = {
    **selection_rule,
    'selection_reason': selection_reason,
    'selected_dual': selected_dict,
}
atomic_json_dump(selection, f'{OUTPUT_DIR}/selected_operating_point.json')
print(json.dumps(selection, indent=2))


In [ ]:
# 19. Independent confirmation comparison at the tune-selected matched operating points.
def per_example_metric(predictions, items, metric):
    return np.array([
        grounding_score(prediction, item['ex']['triples'])[metric]
        for prediction, item in zip(predictions, items)
    ], dtype=float)

def paired_bootstrap(candidate_predictions, baseline_predictions, items, metric,
                     samples=BOOTSTRAP_SAMPLES, seed=20260725):
    difference = per_example_metric(candidate_predictions, items, metric) - per_example_metric(
        baseline_predictions, items, metric)
    rng = np.random.default_rng(seed)
    bootstrap = np.array([
        difference[rng.integers(0, len(difference), len(difference))].mean()
        for _ in range(samples)
    ])
    if metric == 'recall':
        improved = int((difference > 0).sum()); worsened = int((difference < 0).sum())
    else:
        improved = int((difference < 0).sum()); worsened = int((difference > 0).sum())
    return {
        'delta': float(difference.mean()),
        'ci95': [float(np.percentile(bootstrap, 2.5)), float(np.percentile(bootstrap, 97.5))],
        'improved': improved,
        'worsened': worsened,
        'tied': int((difference == 0).sum()),
    }

if RUN_CONFIRMATION:
    CONFIRM_ITEMS = dev_items(GEN_CONFIRM_IDX)
    shared_predictions, shared_triggers = run_cached(
        CONFIRM_ITEMS, 'confirm', 'shared', SHARED_REFERENCE_TAU,
        use_hard=False, save_events=True)
    dual_predictions, dual_triggers = run_cached(
        CONFIRM_ITEMS, 'confirm', SELECTED_DUAL_KEY, SELECTED_DUAL_TAU,
        use_hard=False, save_events=True)

    confirmation_metrics = {
        'shared_reference': {
            'model_key': 'shared', 'tau': SHARED_REFERENCE_TAU,
            **score_block(shared_predictions, CONFIRM_ITEMS, shared_triggers),
        },
        'matched_dual': {
            'model_key': SELECTED_DUAL_KEY, 'tau': SELECTED_DUAL_TAU,
            **score_block(dual_predictions, CONFIRM_ITEMS, dual_triggers),
        },
    }
    confirmation_paired = {
        metric: paired_bootstrap(
            dual_predictions, shared_predictions, CONFIRM_ITEMS, metric,
            seed=20260725 + index)
        for index, metric in enumerate(('recall','halluc'))
    }
    confirmation_paired['changed_outputs'] = int(sum(
        candidate != baseline
        for candidate, baseline in zip(dual_predictions, shared_predictions)
    ))
    confirmation_result = {
        'scientific_status': 'independent development confirmation; repaired test untouched',
        'selection': selection,
        'metrics': confirmation_metrics,
        'paired': confirmation_paired,
    }
    atomic_json_dump(confirmation_result, f'{OUTPUT_DIR}/confirmation_matched_result.json')
    display(pd.DataFrame(confirmation_metrics).T)
    print(json.dumps(confirmation_paired, indent=2))
else:
    print('Confirmation disabled.')


In [ ]:
# 20. Optional secondary confirmation with Hard-v2 using the same selected thresholds.
# This cell performs no tuning and does not affect the main operating-point decision.
if RUN_CONFIRMATION and RUN_HARD_V2_SECONDARY:
    shared_hard_predictions, shared_hard_triggers = run_cached(
        CONFIRM_ITEMS, 'confirm', 'shared', SHARED_REFERENCE_TAU,
        use_hard=True, save_events=True)
    dual_hard_predictions, dual_hard_triggers = run_cached(
        CONFIRM_ITEMS, 'confirm', SELECTED_DUAL_KEY, SELECTED_DUAL_TAU,
        use_hard=True, save_events=True)
    hard_result = {
        'shared_reference_hard_v2': score_block(
            shared_hard_predictions, CONFIRM_ITEMS, shared_hard_triggers),
        'matched_dual_hard_v2': score_block(
            dual_hard_predictions, CONFIRM_ITEMS, dual_hard_triggers),
        'paired': {
            metric: paired_bootstrap(
                dual_hard_predictions, shared_hard_predictions, CONFIRM_ITEMS, metric,
                seed=20260800 + index)
            for index, metric in enumerate(('recall','halluc'))
        },
    }
    hard_result['paired']['changed_outputs'] = int(sum(
        candidate != baseline
        for candidate, baseline in zip(dual_hard_predictions, shared_hard_predictions)
    ))
    atomic_json_dump(hard_result, f'{OUTPUT_DIR}/confirmation_matched_hard_v2_result.json')
    print(json.dumps(hard_result, indent=2))
else:
    print('Hard-v2 secondary confirmation disabled by default.')


In [ ]:
# 21. Package the development-only operating-point analysis.
checkpoint_manifest = {
    key: {'path': path, 'sha256': sha256_file(path), 'epoch': int(torch.load(path, map_location='cpu')['epoch'])}
    for key, path in EPOCH_PATHS.items()
}
run_manifest = {
    'notebook': '19_operating_point_matched_dual_dev_colab.ipynb',
    'scientific_status': 'development-only calibration and independent confirmation; repaired test untouched',
    'fusion_checkpoint': CKPT_FUSION,
    'shared_selector': {'path': CURRENT_SELECTOR_CKPT, 'sha256': sha256_file(CURRENT_SELECTOR_CKPT)},
    'dual_checkpoints': checkpoint_manifest,
    'splits': {
        'selector_validation_n': len(SELECTOR_VAL_IDX),
        'generation_tune_n': len(GEN_TUNE_IDX),
        'generation_confirm_n': len(GEN_CONFIRM_IDX),
        'manifest': CURRENT_SPLIT_MANIFEST,
    },
    'tau_grid': TAU_GRID,
    'shared_reference_tau': SHARED_REFERENCE_TAU,
    'compatibility_margin': COMPATIBILITY_MARGIN,
    'match_tolerance_relative': MATCH_TOLERANCE_REL,
    'selected_dual_key': SELECTED_DUAL_KEY,
    'selected_dual_tau': SELECTED_DUAL_TAU,
    'hard_v2_secondary': RUN_HARD_V2_SECONDARY,
    'test_access': False,
}
atomic_json_dump(run_manifest, f'{OUTPUT_DIR}/run_manifest.json')
archive = shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print('Archive:', archive)
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print(' ', filename)


## Interpretation

The confirmation result supports a representation advantage only when the matched dual point improves generation at approximately the same trigger budget. Outcomes should be interpreted as follows:

- **Dual improves at matched budget:** evidence that the constraint-specific representation adds value after calibration.
- **Dual ties at matched budget:** the earlier deficit was mainly calibration; the extra head remains unnecessary unless it offers another advantage.
- **Dual remains worse at matched budget:** stronger evidence that the residual constraint representation or joint retraining harms downstream decisions.
- **Epoch 2 wins while epoch 1 loses:** direct evidence that F1-based checkpoint selection was misaligned with generation.

Do not use the repaired test to select a new threshold or checkpoint after this notebook.
